In [ ]:
#function that only select atoms within the cylinder
def is_inside_cylinder(position, radius_squared, half_height): :
    if abs(position[0])>radius or  abs(position[1])>radius : #coord défini dans writexyz
        return False
    if abs(position[2])) > half_height:
        return False
    return position[0]**2+ position[1]**2 <=radius_squared
           
def rotation_matrix_from_vector(direction):
    """
    Génère une matrice de rotation R pour aligner l'axe z avec la direction hkl donnée. Ensuite, il faut juste la multiplier par les coordonnées des atomes : https://en.wikipedia.org/wiki/Rodrigues%27_rotation_formula
    - direction : [h, k, l]
    """
    direction = np.array(direction, dtype=float)
    direction = direction / np.linalg.norm(direction)  # normalisation

    # si déjà aligné avec z, pas de rotation nécessaire
    if np.allclose(direction, [0, 0, 1]):
        return np.eye(3)  # Matrice identité

    # axe de rotation (il est égal au produit vectoriel de z(001) et la direction hkl)
    z_axis = np.array([0, 0, 1], dtype=float)
    rotation_axis = np.cross(z_axis, direction)
    rotation_axis = rotation_axis / np.linalg.norm(rotation_axis) # normalisation

    # angle de rotation (formule du produit scalaire)
    angle = np.arccos(np.dot(z_axis, direction))

    # matrice de rotation (formule de Rodrigues)
    K = np.array([[0, -rotation_axis[2], rotation_axis[1]],[rotation_axis[2], 0, -rotation_axis[0]],[-rotation_axis[1], rotation_axis[0], 0]])
    R = np.eye(3) + np.sin(angle) * K + (1 - np.cos(angle)) * np.dot(K, K)
    return R

def makeCylinder(self,noOutput): #in def init, the two entries are specified : size=[diameter,length] and directionCylinder=[h,k,l]
    """
    Create a cylinder along the direction [hkl], [radius,length] are specified by user
    """
    
        if not noOutput: vID.centertxt(f"Removing atoms to make a cylinder",bgc='#cbcbcb',size='12',fgc='b',weight='bold')
        if not noOutput: chrono = pyNMBu.timer(); chrono.chrono_start()
        #dimension
        radius = 10*self.size[0]/2 
        radius_squared=radius**2
        half_height = 10 * self.size[1] / 2
        #creation of the supercell
        self.NP = self.sc.copy() # sc=make_supercell(sc1nm,M)
        if not np.allclose(self.directionCylinder, [0, 0, 1]): # condition if the direction [hkl] given isn't the same as z[0,0,1] 
            R = rotation_matrix_from_vector(self.directionCylinder)
            rotated_positions = np.dot(self.sc.positions, R.T)  # Appliquer la rotation à toutes les coordonnées
        else:
            rotated_positions = self.sc.positions.copy()
    
        # Filtrage des atomes dans le cylindre
        atoms_to_keep = [i for i, pos in enumerate(rotated_positions)if is_inside_cylinder(pos, radius_squared, half_height)]
        self.NP = self.NP[atoms_to_keep]

         # Si rotation, revenir à la configuration originale
        if not np.allclose(self.directionCylinder, [0, 0, 1]):
            self.NP.positions = np.dot(self.NP.positions, np.linalg.inv(R).T) #R est une matrice orthogonale (propriété des matrices de rotation), son inverse est équivalent à sa transposée,

        # nAtoms = self.NP.get_global_number_of_atoms()#d'ou ca sort ?
        if not noOutput: vID.centertxt(f"Nanowire moved to the center of the unitcell",bgc='#cbcbcb',size='12',fgc='b',weight='bold')
        # self.NP.center()
        if not noOutput: chrono.chrono_stop(hdelay=False); chrono.chrono_show()

In [ ]:
import importlib
importlib.reload(pyNMBu)

from pyNanoMatBuilder import crystalNPs as cyNP
importlib.reload(cyNP)
direction = [0,0,1]
dim = [1,6] #radii and length

RuWire = cyNP.Crystal("Au fcc",shape='cylinder',size=dim,directionCylinder=direction,pbc=True,aseView= False)
pyNMBu.writexyz("coords/Aucylinder.xyz", RuWire.NP)
with open("coords/Aucylinder.script", 'w') as f: f.write(RuWire.jMolCS)

RuWire.NPcs.center()
write("coords/Aucylinder_CoreSurface.cif", RuWire.NPcs)
RuWire.NP.center()
write("coords/Aucylinder.cif", RuWire.NP)

In [ ]:
# dans utils rotateMoltoAlignItWithAxis et alignV1WithV2_returnR et  font deja ce que je fais .............................